In [68]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,Conv1D,Flatten,Dense,MaxPool1D,Dropout
import tensorflow as tf

data = pd.read_csv('IMDB Dataset.csv')
data.head()

data = data[:10]
data

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [69]:
def clean_text(text):
    
    # Remove Html
    soup = BeautifulSoup(text)
    text = soup.get_text()
    print(text)

    text = text.lower()
    #Stemmer
    porterStemmer = PorterStemmer()
    
    #Puncatuation
    punctuations = list(string.punctuation)

    # StopWords
    listOfStopword =  stopwords.words('english')
    
    filterSentence = []
    for token in text:
        if (token not in listOfStopword ) and (token not in punctuations):
            word = porterStemmer.stem(token)
            filterSentence.append(word)
            text = ''.join(filterSentence)
             
    return text

data['review'] = data['review'].apply(clean_text)

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wou

In [70]:

from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()
y = label.fit_transform(data['sentiment'])

In [71]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data['review'])
vocab_len = len(tokenizer.word_index)+1

encoded = tokenizer.texts_to_sequences(data['review'])
list_len = []
for seq in encoded:
    list_len.append(len(seq))

max_len = max(list_len)

x = pad_sequences(encoded,maxlen=max_len,padding='post')
x_train, x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)


In [84]:
# reg = tf.keras.regularizers.L2(0.001)
model = Sequential()
# Number of features => output_dim
# Number of words in each sentences
model.add(Embedding(input_dim = vocab_len, output_dim = 50, input_length = max_len, embeddings_regularizer=None ))
# model.add(Dropout(0.2))
model.add(Conv1D(filters = 50, kernel_size=4, padding='same', activation='relu'))
# model.add(Dropout(0.2))
model.add(MaxPool1D(pool_size=2))
model.add(Flatten())
model.add(Dense(64,activation='relu'))
# model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer= tf.optimizers.Adam(),loss= tf.losses.BinaryCrossentropy(), metrics=tf.metrics.BinaryAccuracy())
model.fit(x_train,y_train, batch_size=250, epochs=10, validation_data=(x_test,y_test))

Epoch 1/10
1/1 [==============================] - 2s 2s/step - loss: 0.6971 - binary_accuracy: 0.3750 - val_loss: 0.6447 - val_binary_accuracy: 1.0000
Epoch 2/10
1/1 [==============================] - 0s 63ms/step - loss: 0.6402 - binary_accuracy: 0.6250 - val_loss: 0.5867 - val_binary_accuracy: 1.0000
Epoch 3/10
1/1 [==============================] - 0s 63ms/step - loss: 0.5795 - binary_accuracy: 0.7500 - val_loss: 0.5336 - val_binary_accuracy: 1.0000
Epoch 4/10
1/1 [==============================] - 0s 71ms/step - loss: 0.5150 - binary_accuracy: 1.0000 - val_loss: 0.5038 - val_binary_accuracy: 1.0000
Epoch 5/10
1/1 [==============================] - 0s 60ms/step - loss: 0.4484 - binary_accuracy: 1.0000 - val_loss: 0.4969 - val_binary_accuracy: 1.0000
Epoch 6/10
1/1 [==============================] - 0s 65ms/step - loss: 0.3852 - binary_accuracy: 1.0000 - val_loss: 0.5051 - val_binary_accuracy: 1.0000
Epoch 7/10
1/1 [==============================] - 0s 67ms/step - loss: 0.3249 - bina